# AF — базовое решение на YOLO

Модуль детекции активного горения. Вход — AF-чип 256×256 @ 375 м, выход — бинарная
маска в RLE. Метрика: `F1_af`, попиксельный микро-усреднённый F1 (вес 0.35 в Score).

Подробное описание каждого блока — в `STRUCTURE.md`.

## 1. Конфигурация

In [ ]:
import os, sys, json, random, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
import cv2
import torch
import matplotlib.pyplot as plt
from scipy import ndimage
from ultralytics import YOLO

warnings.filterwarnings("ignore")

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)

ROOT  = Path("af")
TEST  = Path("fire-test-renamed/test")
WORK  = Path("work"); WORK.mkdir(exist_ok=True)
VIIRS, AUX, MASKS = ROOT/"viirs", ROOT/"aux", ROOT/"masks"

DEVICE   = 0 if torch.cuda.is_available() else "cpu"
UPSCALE  = 4
IMGSZ    = 256 * UPSCALE
EPOCHS   = 100
BATCH    = 2
WORKERS  = 0

I4_LO, I4_HI = 270.0, 367.0
D_LO,  D_HI  =   0.0,  60.0
I3_LO, I3_HI =   0.0,   0.6

VAL_TILE_FRAC = 0.30

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "| device:", DEVICE)

## 2. Чтение чипа

In [ ]:
def load_chip(cid, root=ROOT, with_mask=True):
    out = {}
    with rasterio.open(root/"viirs"/f"{cid}_VIIRS_I1-I5.tif") as s:
        for i, name in enumerate(s.descriptions, 1):
            out[name] = s.read(i).astype(np.float32)
    if with_mask:
        with rasterio.open(MASKS/f"{cid}_MASK.tif") as s:
            out["fire"] = (s.read(1) == 1)
    return out

def chip_ids(root=ROOT):
    return sorted(p.name.split("_VIIRS")[0] for p in (root/"viirs").glob("*.tif"))

IDS  = chip_ids()
meta = pd.read_csv(ROOT/"meta.csv")
meta["dt"]    = pd.to_datetime(meta.acq_datetime)
meta["year"]  = meta.dt.dt.year
meta["pos"]   = meta.n_fire_px > 0
meta["tile"]  = list(zip(meta.epsg, meta.x_min, meta.y_min))
meta["scene"] = list(zip(meta.acq_datetime, meta.satellite))

print(f"чипов: {len(IDS)} | с огнём: {int(meta.pos.sum())} | пикселей горения: {int(meta.n_fire_px.sum())}")

## 3. Метрика организаторов

In [ ]:
class MicroF1:
    def __init__(self):
        self.tp = self.fp = self.fn = 0

    def add(self, pred, gt):
        pred = pred.astype(bool); gt = gt.astype(bool)
        self.tp += int(( pred &  gt).sum())
        self.fp += int(( pred & ~gt).sum())
        self.fn += int((~pred &  gt).sum())
        return self

    @property
    def precision(self):
        return self.tp / (self.tp + self.fp) if (self.tp + self.fp) else 1.0

    @property
    def recall(self):
        return self.tp / (self.tp + self.fn) if (self.tp + self.fn) else 1.0

    @property
    def f1(self):
        p, r = self.precision, self.recall
        return 2*p*r/(p+r) if (p + r) else 0.0

    def __repr__(self):
        return f"P={self.precision:.4f} R={self.recall:.4f} F1={self.f1:.4f}"

_g = np.zeros((256,256), bool); _g[10:14, 20:24] = True
assert abs(MicroF1().add(_g, _g).f1 - 1.0) < 1e-12
assert MicroF1().add(np.zeros_like(_g), _g).f1 == 0.0
print("метрика: тесты пройдены")

## 4. Потолок box-подхода

In [ ]:
STRUCT = np.ones((3,3), int)

def components(mask):
    lab, k = ndimage.label(mask, structure=STRUCT)
    return lab, k, ndimage.find_objects(lab)

oracle = MicroF1()
for cid in IDS:
    gt = load_chip(cid)["fire"]
    pred = np.zeros_like(gt)
    for sl in components(gt)[2]:
        pred[sl] = True
    oracle.add(pred, gt)

CEILING = oracle.f1
print("оракул bbox-заливки:", oracle)
print(f"ПОТОЛОК F1_af для любого box-детектора = {CEILING:.4f}")

## 5. Рендер: 13 каналов -> 3 канала uint8

In [ ]:
def _scale(a, lo, hi):
    a = np.nan_to_num(a, nan=lo, posinf=hi, neginf=lo)
    return (np.clip((a - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)

def render(chip):
    r = _scale(chip["I4"],              I4_LO, I4_HI)
    g = _scale(chip["I4"] - chip["I5"], D_LO,  D_HI)
    b = _scale(chip["I3"],              I3_LO, I3_HI)
    inv = chip["valid"] < 0.5
    r[inv] = g[inv] = b[inv] = 0
    img = np.dstack([b, g, r])
    return cv2.resize(img, (IMGSZ, IMGSZ), interpolation=cv2.INTER_NEAREST)

print("рендер:", render(load_chip(IDS[0])).shape)

## 6. Разметка: маски -> bbox

In [ ]:
def mask_to_boxes(mask):
    if not mask.any():
        return []
    H, W = mask.shape
    out = []
    for sl in components(mask)[2]:
        y0, y1 = sl[0].start, sl[0].stop
        x0, x1 = sl[1].start, sl[1].stop
        out.append(((x0+x1)/2/W, (y0+y1)/2/H, (x1-x0)/W, (y1-y0)/H))
    return out

_t = np.zeros((256,256), bool); _t[100, 50] = True
assert abs(mask_to_boxes(_t)[0][2] - 1/256) < 1e-9, "объект 1px получил нулевую ширину"
print("разметка: тест пройден")

## 7. Сплит train/val с блокировкой по территории

In [ ]:
tiles = sorted(meta.tile.unique(), key=str)
_rng  = np.random.RandomState(SEED)
_order = list(tiles); _rng.shuffle(_order)
val_tiles = set(_order[:int(round(VAL_TILE_FRAC*len(_order)))])

is_val   = meta.tile.isin(val_tiles)
is_train = ~is_val
train_ids = meta.loc[is_train, "chip_id"].tolist()
val_ids   = meta.loc[is_val,   "chip_id"].tolist()

tr, va = meta.loc[is_train], meta.loc[is_val]
assert not (set(tr.tile) & set(va.tile)), "тайлы пересекаются -> территориальная утечка"

print(f"train {len(train_ids)} чипов ({int(tr.pos.sum())} с огнём, {int(tr.n_fire_px.sum())} px)")
print(f"val   {len(val_ids)} чипов ({int(va.pos.sum())} с огнём, {int(va.n_fire_px.sum())} px)")
print(f"общих тайлов train/val: {len(set(tr.tile) & set(va.tile))}")
print(f"общих сцен  train/val: {len(set(tr.scene) & set(va.scene))}")

## 8. Экспорт в формат ultralytics

In [ ]:
DS = WORK/"yolo_ds"
if DS.exists():
    shutil.rmtree(DS)
for sub in ["images/train", "images/val", "labels/train", "labels/val"]:
    (DS/sub).mkdir(parents=True)

def export(ids, split):
    n = 0
    for cid in ids:
        chip = load_chip(cid)
        cv2.imwrite(str(DS/f"images/{split}/{cid}.png"), render(chip))
        boxes = mask_to_boxes(chip["fire"]); n += len(boxes)
        (DS/f"labels/{split}/{cid}.txt").write_text(
            "".join(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n" for xc, yc, w, h in boxes))
    return n

n_tr = export(train_ids, "train")
n_va = export(val_ids,   "val")
(DS/"data.yaml").write_text(
    f"path: {DS.resolve()}\ntrain: images/train\nval: images/val\nnc: 1\nnames: [fire]\n")
print(f"train: {len(train_ids)} изображений, {n_tr} боксов")
print(f"val  : {len(val_ids)} изображений, {n_va} боксов")

## 9. Обучение

In [ ]:
RUN  = WORK/"runs"/"af_yolo"
BEST = RUN/"weights"/"best.pt"

if BEST.exists():
    print("используем готовые веса:", BEST)
else:
    model = YOLO("yolo11n.pt")
    model.train(
        data=str((DS/"data.yaml").resolve()),
        epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
        seed=SEED, deterministic=True, workers=WORKERS,
        project=str((WORK/"runs").resolve()), name="af_yolo", exist_ok=True,
        val=True, plots=True, verbose=True,

        hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
        degrees=0.0, translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
        flipud=0.0, fliplr=0.0, mosaic=0.0, mixup=0.0, copy_paste=0.0, erasing=0.0,
    )
    BEST = Path(model.trainer.save_dir)/"weights"/"best.pt"

print("веса:", BEST)

## 10. Оценка по метрике организаторов

In [ ]:
def boxes_to_mask(res, conf, shape=(256,256)):
    out = np.zeros(shape, bool)
    if res.boxes is None or len(res.boxes) == 0:
        return out
    xyxy = res.boxes.xyxy.cpu().numpy() / UPSCALE
    cf   = res.boxes.conf.cpu().numpy()
    for (x0, y0, x1, y1), p in zip(xyxy, cf):
        if p < conf:
            continue
        out[max(int(np.floor(y0)),0):min(int(np.ceil(y1)),shape[0]),
            max(int(np.floor(x0)),0):min(int(np.ceil(x1)),shape[1])] = True
    return out

model = YOLO(str(BEST))
preds = [model.predict(str(DS/f"images/val/{cid}.png"), imgsz=IMGSZ, conf=0.001,
                       device=DEVICE, verbose=False)[0] for cid in val_ids]
gts   = [load_chip(cid)["fire"] for cid in val_ids]

rows = []
for t in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
    mt = MicroF1()
    for r, g in zip(preds, gts):
        mt.add(boxes_to_mask(r, t), g)
    rows.append((t, mt.precision, mt.recall, mt.f1))

tab = pd.DataFrame(rows, columns=["conf", "precision", "recall", "F1_af"])
BEST_CONF = float(tab.loc[tab.F1_af.idxmax(), "conf"])
BASE_F1   = float(tab.F1_af.max())
print(tab.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"\nлучший conf = {BEST_CONF}  ->  F1_af = {BASE_F1:.4f}")

In [ ]:
zero = MicroF1()
for g in gts:
    zero.add(np.zeros_like(g), g)

print("=" * 58)
print(f"{'потолок box-подхода (оракул)':<40} {CEILING:.4f}")
print(f"{'YOLO-det бейзлайн':<40} {BASE_F1:.4f}")
print(f"{'тривиальный ответ (всё ноль)':<40} {zero.f1:.4f}")
print("=" * 58)
print(f"реализовано от потолка: {100*BASE_F1/CEILING:.1f} %")

## 11. RLE

In [ ]:
def rle_encode(mask):
    flat = np.concatenate([[0], mask.ravel(order="C").astype(np.uint8), [0]])
    ch = np.flatnonzero(flat[1:] != flat[:-1]) + 1
    return " ".join(f"{s} {e-s}" for s, e in zip(ch[0::2], ch[1::2]))

def rle_decode(s, shape=(256,256)):
    out = np.zeros(shape[0]*shape[1], bool)
    if isinstance(s, str) and s.strip():
        v = list(map(int, s.split()))
        for st, ln in zip(v[0::2], v[1::2]):
            out[st-1 : st-1+ln] = True
    return out.reshape(shape, order="C")

bad = sum(1 for cid in IDS
          if not np.array_equal(rle_decode(rle_encode(load_chip(cid)["fire"])),
                                load_chip(cid)["fire"]))
assert bad == 0
assert rle_encode(np.zeros((256,256), bool)) == ""
print(f"RLE round-trip: {len(IDS)}/{len(IDS)} совпало")

## 12. Инференс на тесте -> submission

In [ ]:
tmpl = pd.read_csv(TEST/"sample_submission.csv")
af_chips = tmpl[tmpl.chip_id.str.startswith("AF_")].chip_id.unique().tolist()
print(f"строк в шаблоне: {len(tmpl)} | AF-чипов: {len(af_chips)}")

rows = []
for i, cid in enumerate(af_chips, 1):
    chip = load_chip(cid, root=TEST/"af", with_mask=False)
    res  = model.predict(render(chip), imgsz=IMGSZ, conf=0.001, device=DEVICE, verbose=False)[0]
    rows.append({"chip_id": cid, "class_id": 1,
                 "rle": rle_encode(boxes_to_mask(res, BEST_CONF))})
    if i % 60 == 0 or i == len(af_chips):
        print(f"  {i}/{len(af_chips)}", flush=True)

sub = pd.DataFrame(rows, columns=["chip_id", "class_id", "rle"])
sub["rle"] = sub["rle"].fillna("").astype(str)
assert sub.notna().all().all()
assert (sub.class_id == 1).all()
assert set(sub.chip_id) == set(af_chips)
sub.to_csv("submission_yolo.csv", index=False)

n_px = sum(sum(map(int, r.split()[1::2])) for r in sub.rle if r)
print(f"\nзаписано submission_yolo.csv: {len(sub)} строк")
print(f"непустых масок: {int((sub.rle.str.len()>0).sum())} / {len(sub)}")
print(f"пикселей горения предсказано: {n_px} ({n_px/len(sub):.0f} на чип)")